In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("avalanche_dataset/warnings_winter_2023_2026.csv")
risk_order = ["Low", "Medium", "High"]
df["risk"] = pd.Categorical(df["risk"], categories=risk_order, ordered=True)
colors = {"Low": "mediumseagreen", "Medium": "orangered", "High": "maroon"}
month_order = [12, 1, 2, 3, 4, 5]

In [ ]:
# Risk proportion per month

ct_m = pd.crosstab(df["publish_month"], df["risk"]).reindex(month_order)
pr_m = ct_m.div(ct_m.sum(axis=1), axis=0)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
pr_m.plot(kind="bar", stacked=True, ax=ax[0], color=[colors[c] for c in risk_order])
ax[0].set_title("Risk distribution per month (proportion)")
ax[0].set_xlabel("Month"); ax[0].tick_params(axis="x", rotation=0)

In [ ]:
# Risk proportion per region

ct_r = pd.crosstab(df["geographical_region"], df["risk"])
pr_r = ct_r.div(ct_r.sum(axis=1), axis=0).sort_values("High")
labels = [f"{r} (n={ct_r.loc[r].sum()})" for r in pr_r.index]
pr_r.plot(kind="barh", stacked=True, ax=ax[1],
          color=[colors[c] for c in risk_order])
ax[1].set_yticklabels(labels); ax[1].set_xlabel("Proportion")
ax[1].set_title("Risk distribution per geographical region")

In [ ]:
plt.tight_layout(); plt.show()

In [ ]:
# Heatmap for numerical features (excluding temporal)

num_cols = ["rainfall_most_exposed_area", "rainfall_average",
            "temperature_min", "temperature_max",
            "freezing_level_masl", "rain_snow_line_altitude_masl",
            "wind_direction_change"]

# ordinal encoding of the target, only for the correlation
df["risk_num"] = df["risk"].cat.codes  

corr = df[num_cols].corr(method="pearson")
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(7, 6))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, square=True)
plt.title("Pearson correlation between numerical predictors")
plt.tight_layout(); plt.show()